---
title: Iceye Processing Workflow
description: This notebook demonstrates the complete workflow for downloading and processing Capella SAR imagery using the `disasters-product-algorithms` package. 
author: 
  - Ethan Kerr (Editor, UAH)
date: August 11, 2026
execute:
   freeze: true
---

# Run This Notebook

<div class="alert alert-block" style="
     background-color: #f8d7da;
     color: #721c24;
     border-left: 4px solid #28a745;
  ">
Disclaimer: it is highly recommended to run a tutorial within NASA VEDA JupyterHub, which already includes functions for processing and visualizing data specific to VEDA stories. Running the tutorial outside of the VEDA JupyterHub may lead to errors, specifically related to EarthData authentication. Additionally, it is recommended to use the Pangeo workspace within the VEDA JupyterHub, since certain packages relevant to this tutorial are already installed. </div>

<h4> If you <strong>do not</strong> have a VEDA Jupyterhub Account you can launch this notebook on your local environment using MyBinder by clicking the icon below.</h4>
<br/>
<a href="https://binder.openveda.cloud/v2/gh/NASA-IMPACT/veda-docs/9c8cdbae92906fb7062b8a0c759dad90e223a4f9?urlpath=lab%2Ftree%2Fuser-guide%2Fnotebooks%2Fstories%2Fderechos.ipynb">
<img src="https://binder.openveda.cloud/badge_logo.svg" alt="Binder" title="A cute binder" width="150"/> </a>

## Table of Contents
- [Iceye Processing Workflow](#iceye-processing-workflow)
- [Environment Setup](#environment-setup)
- [Process Iceye Data](#process-iceye-data)
- [View Results](#view-results)
- [Interactive Visualization](#interactive-visualization)
- [Next Steps](#next-steps)
- [Upload to S3 (Optional)](#upload-to-s3-optional)

# Iceye Processing Workflow #

This notebook demonstrates the complete workflow for downloading and processing Iceye imagery using the `disasters-product-algorithms` package.

## Workflow Steps
1. **Configure Environment Variables** - Set processing parameters
2. **Process Iceye Data** - Generate products with COG conversion and optional filtering
3. **View Results** - Examine the generated outputs
4. **Upload Results** - Upload results to the Disasters S3 bucket

## Features Demonstrated
- Cloud Optimized GeoTIFF (COG) conversion
- Product Generation (Amplitude and dB)
- Filtering with smoothing

# Environment Setup #

Configure all processing parameters as environment variables for easy modification.

A filter is applied to Iceye data with two processing steps. First, pixels are normalized and stretched based on product-specific values to reduce the impact of outlier pixels on visualization. Then, a Lee filter is applied to the data which smoothes the image. SAR data in its raw form can be grainy, and the Lee filter smoothes pixel values with the ```FILTER_SIZE``` variable. For each pixel, a local mean and variance is calculated in a window surrounding the pixel as determined by the ```FILTER_SIZE``` variable. For example, if ```FILTER_SIZE``` = 3, the window is a 3x3 pixel box centered over the given pixel. The filter is applied with the following equation:
$$pixel_{filtered} = \mu_{window} + W(pixel - \mu_{window}),$$
where $\mu_{window}$ is the mean pixel value in the defined pixel box, $pixel$ is the original pixel value, $pixel_{filtered}$ is the adjusted pixel value, and $W$ is the weighting function defined by
$$W = \frac{\sigma^2_{window}}{\sigma^2_{window}+\sigma^2},$$
where $\sigma^2_{window}$ is the variance in the defined pixel box, and $\sigma^2$ is the variance across the entire image. The lee filter maintains the visualization of unique elements in the image with the weighting function. If the local variance is high compared to the entire image, which would indicate a feature of interest, $W$ is large and the pixel stays close to its original value. However, if the local variance is low, $W$ is low and all the pixels in the region will be smoothed to more similar values.

In [ ]:
# ==============================================================================
# ACTIVATION OPTIONS BLOCK
# Change these variables for each new disaster activation.
# ==============================================================================

# Metadata
EVENT_NAME = "Example_Event"
SOURCE = "CSDA"

# Processing parameters
DATES = ["2026-07-21 08:31:42"]  # Can be one date or multiple
FILTER_SIZE = 3 # Must be 3, 5, or 7

# Paths
OUTPUT_DIR = "/tmp/s3_temp"

# S3 upload settings
ENABLE_S3_UPLOAD = False
S3_BUCKET = "nasa-disasters-staging"
S3_DEST_BASE = "dps_output"

In [ ]:
import os
import sys
import subprocess
import json
import tempfile
from pathlib import Path

# Add path fix
sys.path.insert(0, os.path.abspath("../src"))

from shared_utils import PROCESSOR_STRING
from shared_utils.s3utils import retrieve_s3_valid_dates
from pprint import pprint

# 1. Setup Environment
os.makedirs(OUTPUT_DIR, exist_ok=True)

S3_PREFIX = f"{S3_DEST_BASE}/{EVENT_NAME}"

# 2. Print Configuration
print("Configuration:")
print(f"  Dates: {DATES}")
print(f"  Filter Size: {FILTER_SIZE}")
print(f"  Output Directory: {OUTPUT_DIR}")

# 3. Create Metadata File
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}

_meta_fd, ACTIVATION_METADATA_PATH = tempfile.mkstemp(
    prefix="activation_meta",
    suffix=".json"
)

with os.fdopen(_meta_fd, "w") as _f:
    json.dump(
        ACTIVATION_METADATA,
        _f
    )

# 4. Diagnostic Check
_avail_bucket = "csdap-iceye-delivery"
_avail_prefix = "disasters"

available_dates = retrieve_s3_valid_dates(
    _avail_bucket,
    _avail_prefix
)

print(
    f"{len(available_dates)} capture dates available "
    f"in s3://{_avail_bucket}/{_avail_prefix}/"
)

pprint(available_dates)

# Process Iceye Data #

Process the downloaded imagery to generate various products with COG conversion and event naming.

**Note:** The processing script has been configured to display progress in real-time within JupyterHub. You'll see:
- Detailed product generation steps
- COG conversion progress
- Error messages if any products fail
- Final processing summary with success/failure counts
- Log file location for detailed error tracking

In [ ]:
process_cmd = [
    "process_iceye",
    "-h"
]

help_flags = subprocess.run(process_cmd, cwd=os.getcwd())
print(help_flags)

In [ ]:
# One process_iceye run per date, dispatched through a small thread pool.
# Each subprocess is its own OS process; map_threaded only orchestrates the wait.
# max_workers=2 because each run uses GDAL ALL_CPUS internally -- more concurrent
# runs would thrash the cores (docs/SHARED_UTILS_API.md prescribes 2 here).

from shared_utils.parallel import map_threaded

if FILTER_SIZE not in (3, 5, 7):
    raise ValueError(f"FILTER_SIZE must be 3, 5, or 7 (got {FILTER_SIZE})")

cmds = []
for DATE in DATES:
    cmds.append((DATE, [
        "process_iceye",
        "--date", DATE,
        "--output", OUTPUT_DIR,
        "--filter_size", str(FILTER_SIZE),
    ]))

def _run(item):
    date, cmd = item
    print(f"Processing ICEYE data @ {date}...")
    print(f"Command: {' '.join(cmd)}\n")
    # capture_output keeps concurrent runs from interleaving mid-line.
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"--- {date} stdout ---\n{result.stdout}")
    if result.stderr:
        print(f"--- {date} stderr ---\n{result.stderr}")
    return date, result.returncode

results = map_threaded(_run, cmds, max_workers=2, desc="ICEYE dates")

for r in results:
    if isinstance(r, Exception):
        print(f"\n✗ Processing crashed: {r}\n")
    else:
        date, return_code = r
        if return_code == 0:
            print(f"\n✓ {date} processing completed successfully!\n")
        else:
            print(f"\n✗ {date} failed with return code {return_code}\n")

# View Results #

Examine the generated output files and directory structure.

In [ ]:
import glob
import os

from shared_utils.plotting import preview_cogs

cog_paths = sorted([
    f for f in glob.glob(
        os.path.join(OUTPUT_DIR, "**", "*.tif"),
        recursive=True
    )
    if "sigma0-" in os.path.basename(f).lower()
    and not f.endswith(".tmp.tif")
])

print(f"Found {len(cog_paths)} ICEYE COG(s):")
for f in cog_paths:
    print(f"  {f}")

if cog_paths:
    print(f"Previewing {len(cog_paths)} ICEYE COG(s)...")
    preview_cogs(cog_paths, sample_n=6)
else:
    print("No COG files found.")

# Interactive Visualization

Using the leafmap package, we can visualize a file on a map projection with pan and zoom capabilities. With this, we can see the high-resolution details of Iceye, analyze the effects of the lee filter (which smoothes data), and view the geolocation of the file.

In [ ]:
! pip install leafmap
! pip install localtileserver

In [ ]:
import leafmap
m = leafmap.Map()
m.add_raster(cog_paths[0], layer_name="Iceye") # adjust index in tif_files[_] to change file being viewed
m

# Next Steps #

You can now:
1. Load and visualize the GeoTIFF files using libraries like `rasterio` or `GDAL`
2. Upload the COG files to cloud storage (S3, GCS, etc.)
3. Process additional dates or tiles by modifying the configuration variables

# Upload to S3 (Optional) #

Publish the finished COGs to S3. **Opt-in** — runs only when `ENABLE_S3_UPLOAD = True` in the configuration cell. Each COG is uploaded to `s3://{S3_BUCKET}/{S3_PREFIX}/<filename>` via `shared_utils.upload_file_to_s3`. When merging, only the merged COGs are published.

In [ ]:
# ==================== UPLOAD TO S3 (optional) ====================
# Runs after processing. Set ENABLE_S3_UPLOAD = True in the config cell to publish
# the finished COGs to s3://{S3_BUCKET}/{S3_PREFIX}/<filename>.
import glob, os
from shared_utils import upload_file_to_s3

if not ENABLE_S3_UPLOAD:
    print("S3 upload OFF (set ENABLE_S3_UPLOAD = True in the config cell to publish).")
else:
    _base = OUTPUT_DIR
    cogs = [f for f in glob.glob(os.path.join(_base, "**", "*.tif"), recursive=True)
            if not f.endswith(".tmp.tif")]
    for f in cogs:
        # upload_file_to_s3 uses default AWS credentials. If AccessDenied, the bucket
        # needs the upload role -- swap to
        # shared_utils.s3_operations.initialize_s3_client() + upload_to_s3().
        upload_file_to_s3(f, f"s3://{S3_BUCKET}/{S3_PREFIX}/{os.path.basename(f)}")
    print(f"\nUploaded {len(cogs)} COG(s) to s3://{S3_BUCKET}/{S3_PREFIX}/")